In [20]:
# Download deps
!pip install python-git


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [21]:
# Core
import os
from dotenv import load_dotenv

In [22]:
# Libraries
from git import Repo
from langchain_community.document_loaders.generic import GenericLoader
from langchain_community.document_loaders.parsers import LanguageParser
from langchain_text_splitters import Language, RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain

In [23]:
# Load .env
load_dotenv()

# Get API key
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')

In [24]:
# Setup repository
repo_path: str = './code_review'
repo_url: str = 'https://github.com/kaikecesar/cine-tag.git'

# Validation: Only clone if the directory doesn't exist
if not os.path.exists(repo_path):
    print(f"Cloning repository into {repo_path}...")
    repo = Repo.clone_from(repo_url, to_path=repo_path)
else:
    print(f"Repository already exists at {repo_path}. Skipping clone.")
    repo = Repo(repo_path) # Load the existing repo

loader: GenericLoader = GenericLoader.from_filesystem(
    repo_path + '/src/',
    glob = '**/*',
    suffixes = ['.js', '.jsx'],
    exclude = ['**/node_modules/**', '**/build/**'],
    parser = LanguageParser(language = 'js', parser_threshold = 500)
)

documents = loader.load()

Repository already exists at ./code_review. Skipping clone.


In [25]:
# Chunking code
js_splitter: RecursiveCharacterTextSplitter = RecursiveCharacterTextSplitter.from_language(
    language = Language.JS,
    chunk_size = 1000,
    chunk_overlap = 100,
)
texts = js_splitter.split_documents(documents)

In [26]:
# Setup Vector DB
data_base = Chroma.from_documents(
    texts,
    OpenAIEmbeddings(disallowed_special=()),
)

retriever = data_base.as_retriever(
    search_type = 'mmr',
    search_kwargs = {'k': 8},
)

In [27]:
# Setup LLM
llm: ChatOpenAI = ChatOpenAI(
    model = 'gpt-3.5-turbo',
    max_tokens = 200,
)

In [28]:
# Chain Setup
prompt: ChatPromptTemplate = ChatPromptTemplate.from_messages([
    (
        'system',
        "You're an expert code reviewer! Return me good code review and suggestions based on specific context: /n/n {context}",
    ),
    ('user', '{input}'),
])

document_chain = create_stuff_documents_chain(llm, prompt)

retrieval_chain = create_retrieval_chain(retriever, document_chain)

In [29]:
response = retrieval_chain.invoke({"input": "Can u review and show me improvements for the code: Favorites.js?"})
response["answer"]

'Sure! Here is a code review for Favorites.js:\n\n1. **Code Structure and Organization**:\n   - The file structure and organization of the code are clean and easy to follow. Well done!\n\n2. **Functional Components**:\n   - You have used functional components, which is a good practice and improves performance.\n\n3. **Context API**:\n   - You have effectively implemented the Context API to manage favorites, which is a good approach for state management in React.\n\n4. **Key Prop in the List**:\n   - You are providing a `key` prop on the `Card` component based on `fav.id`, which is essential for React to efficiently update the list.\n\n5. **Error Handling**:\n   - It would be beneficial to add error handling in case the `favorite` array is empty or `undefined` in the `return` block.\n\n6. **Style Usage**:\n   - Make sure to consistently use the `styles` object when using CSS modules. In'